# Exploratory threshold perturbation analysis

This notebook runs a small requested sensitivity analysis for threshold-dependent diagnostic findings. It deliberately changes only numeric thresholds within otherwise similar finding text, queries GPT-5 once per variant, and summarizes whether the estimated likelihood ratios change monotonically.

This is an exploratory sensitivity check, not evidence that the model is reasoning or that memorization/distributional familiarity has been excluded.

## 1. Setup

Outputs are written to a dated `results/YYYY-MM-DD/threshold_perturbation/` folder. The notebook is self-contained and copies the minimal estimator helper code used by the original LR estimator notebook.

In [ ]:
from __future__ import annotations

from datetime import date, datetime, timezone
import json
import math
import os
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

WORKING_DIR = Path.cwd()
RESULTS_ROOT = WORKING_DIR / "results"
OUTPUT_DIR = RESULTS_ROOT / date.today().strftime("%Y-%m-%d") / "threshold_perturbation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(WORKING_DIR / ".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Create .env with OPENAI_API_KEY=... or export it before running."
    )

client = OpenAI(api_key=OPENAI_API_KEY)
MODELS = ["gpt-5"]

print(f"Working directory: {WORKING_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Models: {MODELS}")

## 2. GPT-5 LR estimator

The prompt and structured-output wrapper are copied from the main estimator notebook, limited here to the GPT-5 configuration needed for this supplemental analysis.

In [ ]:
MODEL_CAPABILITIES = {
    "gpt-5": {"reasoning": True, "verbosity": True, "allow_temp": False},
}

SYSTEM_CORE = """You are a Bayesian diagnostic assistant.
Estimate a numeric likelihood ratio (LR) for a finding with respect to a diagnosis.
Return only a JSON object matching the schema: {"value": <float>}, where value > 0.
"""

DEFINITION = """Definition:
LR = P(finding | diagnosis) / P(finding | not-diagnosis)
"""

BANDS = """LR evidence bands (reference):
>10 strong for; 5-10 moderate for; 2-5 weak for;
0.5-2 negligible;
0.2-0.5 weak against; 0.1-0.2 moderate against; <=0.1 strong against"""

FEW_SHOT_MIN = [
    ("deep vein thrombosis", "femoral vein noncompressible on ultrasound", 16.0),
    ("myocardial infarction", "enjoys playing chess", 1.0),
]


def build_messages(diagnosis: str, finding: str, reasoning: bool) -> list[dict]:
    msgs: list[dict] = [
        {"role": "system", "content": SYSTEM_CORE.strip()},
        {"role": "system", "content": DEFINITION.strip()},
        {"role": "system", "content": BANDS.strip()},
    ]
    examples = FEW_SHOT_MIN
    for dx_ex, f_ex, v_ex in examples:
        msgs.append({"role": "user", "content": f"Condition: {dx_ex}\nFinding: {f_ex}"})
        msgs.append({"role": "assistant", "content": f'{{"value": {float(v_ex)}}}'})
    msgs.append({"role": "user", "content": f"Condition: {diagnosis}\nFinding: {finding}"})
    return msgs


class LRResponse(BaseModel):
    value: float


def estimate_lr(diagnosis: str, finding: str, model: str, client: Optional[OpenAI] = None) -> float:
    if client is None:
        client = OpenAI(api_key=OPENAI_API_KEY)

    cfg = MODEL_CAPABILITIES[model]
    msgs = build_messages(diagnosis, finding, reasoning=cfg["reasoning"])

    kwargs = {}
    if cfg["reasoning"]:
        kwargs["reasoning"] = {"effort": "medium"}
    elif cfg["allow_temp"]:
        kwargs["temperature"] = 0.2

    if cfg["verbosity"]:
        kwargs["text"] = {"verbosity": "low"}

    resp = client.responses.parse(
        model=model,
        input=msgs,
        text_format=LRResponse,
        **kwargs,
    )
    value = float(resp.output_parsed.value)
    if not math.isfinite(value) or value <= 0:
        raise ValueError(f"Non-positive or non-finite LR returned: {value!r}")
    return value

## 3. Perturbation cases

The D-dimer family is a deliberately constructed constructed exemplar because the manuscript workbook contains D-dimer rows without numeric thresholds. The remaining families use threshold-sensitive findings represented in the manuscript workbook.

In [ ]:
PERTURBATION_CASES = [
    {
        "case_id": "dvt_d_dimer_constructed",
        "family": "DVT D-dimer",
        "condition": "Deep Venous Thrombosis (DVT)",
        "finding_family": "High-sensitivity D-dimer above threshold in a low pre-test probability Wells patient",
        "case_source": "constructed_threshold_exemplar",
        "source_note": "Workbook contains high-sensitivity D-dimer rows by Wells stratum but without numeric thresholds.",
        "comparator": ">",
        "threshold_units": "ng/mL",
        "thresholds": [500, 550, 1000],
        "finding_template": "Patient has: high-sensitivity D-dimer >{threshold} ng/mL (in a patient known to have a low pre-test probability by Wells)",
    },
    {
        "case_id": "hf_bnp_no_chronic_respiratory_disease",
        "family": "Heart failure BNP",
        "condition": "Dyspnea due to heart failure (in a patient without chronic respiratory disease)",
        "finding_family": "BNP above threshold",
        "case_source": "workbook_grounded_variant",
        "source_note": "Workbook includes BNP >=50, >=100, >=150, >=200, and >=250 rows for this condition.",
        "comparator": ">=",
        "threshold_units": "pg/mL",
        "thresholds": [50, 100, 250],
        "finding_template": "Patient has: BNP >={threshold} pg/mL",
    },
    {
        "case_id": "temporal_arteritis_esr",
        "family": "Temporal arteritis ESR",
        "condition": "Temporal Arteritis",
        "finding_family": "ESR above threshold",
        "case_source": "workbook_grounded_variant",
        "source_note": "Workbook includes ESR >50 and ESR >100 rows for temporal arteritis; >70 is an interpolated perturbation.",
        "comparator": ">",
        "threshold_units": "mm/h",
        "thresholds": [50, 70, 100],
        "finding_template": "Patient has: ESR >{threshold} mm/h",
    },
    {
        "case_id": "cardiac_syncope_troponin_t",
        "family": "Cardiac syncope hs-troponin T",
        "condition": "Cardiac Syncope",
        "finding_family": "High-sensitivity cardiac troponin T above threshold",
        "case_source": "workbook_grounded_variant",
        "source_note": "Workbook includes high-sensitivity cardiac troponin T >42 pg/mL for cardiac syncope; adjacent thresholds are perturbations.",
        "comparator": ">",
        "threshold_units": "pg/mL",
        "thresholds": [30, 42, 60],
        "finding_template": "Patient has: high-sensitivity cardiac troponin T >{threshold} pg/mL",
    },
    {
        "case_id": "elevated_icp_midline_shift_ct",
        "family": "Elevated ICP CT midline shift",
        "condition": "Elevated Intracranial Pressure",
        "finding_family": "CT midline shift above threshold",
        "case_source": "workbook_grounded_variant",
        "source_note": "Workbook includes midline shift >10 mm on CT; >5 and >15 mm are perturbations.",
        "comparator": ">",
        "threshold_units": "mm",
        "thresholds": [5, 10, 15],
        "finding_template": "Patient has: midline shift >{threshold} mm on CT",
    },
]

case_rows = []
for case in PERTURBATION_CASES:
    for variant_order, threshold in enumerate(case["thresholds"], start=1):
        finding_text = case["finding_template"].format(threshold=threshold)
        case_rows.append(
            {
                "case_id": case["case_id"],
                "family": case["family"],
                "condition": case["condition"],
                "finding_family": case["finding_family"],
                "variant_order": variant_order,
                "threshold_value": threshold,
                "threshold_units": case["threshold_units"],
                "comparator": case["comparator"],
                "finding_text": finding_text,
                "expected_direction": "nondecreasing",
                "case_source": case["case_source"],
                "source_note": case["source_note"],
            }
        )

cases_df = pd.DataFrame(case_rows)
cases_path = OUTPUT_DIR / "threshold_perturbation_cases.csv"
cases_df.to_csv(cases_path, index=False)

print(f"Saved cases -> {cases_path}")
display(cases_df)

## 4. Query GPT-5 once per variant

This cell performs the API calls. It writes raw results even if a call fails, then stops before summary generation so failures are not silently analyzed.

In [ ]:
query_rows = []
queried_at = datetime.now(timezone.utc).isoformat()

for model in MODELS:
    for row in cases_df.itertuples(index=False):
        print(f"Querying {model}: {row.family} | {row.finding_text}")
        try:
            lr_estimate = estimate_lr(
                diagnosis=row.condition,
                finding=row.finding_text,
                model=model,
                client=client,
            )
            error_status = "ok"
            error_message = ""
        except Exception as exc:
            lr_estimate = np.nan
            error_status = "error"
            error_message = str(exc)

        query_rows.append(
            {
                **row._asdict(),
                "model": model,
                "lr_estimate": lr_estimate,
                "error_status": error_status,
                "error_message": error_message,
                "queried_at": queried_at,
            }
        )

raw_df = pd.DataFrame(query_rows)
raw_path = OUTPUT_DIR / "threshold_perturbation_raw.csv"
raw_df.to_csv(raw_path, index=False)

print(f"Saved raw results -> {raw_path}")
display(raw_df)

failed = raw_df[raw_df["error_status"].ne("ok")].copy()
if not failed.empty:
    raise RuntimeError(
        "At least one perturbation query failed. Raw results were written; fix the error before interpreting outputs."
    )

invalid = raw_df[~np.isfinite(raw_df["lr_estimate"]) | (raw_df["lr_estimate"] <= 0)].copy()
if not invalid.empty:
    raise RuntimeError(
        "At least one perturbation query returned a missing, non-finite, or non-positive LR."
    )

## 5. Summarize monotonicity and anchoring

Monotonicity is descriptive. A monotonic response supports threshold sensitivity but does not prove reasoning or exclude memorization.

In [ ]:
def classify_monotonicity(values: list[float]) -> str:
    arr = np.asarray(values, dtype=float)
    diffs = np.diff(arr)
    if np.all(diffs == 0):
        return "flat"
    if np.all(diffs > 0):
        return "strictly_increasing"
    if np.all(diffs >= 0):
        return "nondecreasing_with_ties"
    return "nonmonotonic"


def format_sequence(values: list[float], digits: int = 3) -> str:
    formatted = []
    for value in values:
        value = float(value)
        if abs(value) >= 10:
            formatted.append(f"{value:.1f}")
        elif abs(value) >= 1:
            formatted.append(f"{value:.2f}".rstrip("0").rstrip("."))
        else:
            formatted.append(f"{value:.3f}".rstrip("0").rstrip("."))
    return "; ".join(formatted)


def summarize_group(group: pd.DataFrame) -> dict:
    ordered = group.sort_values("variant_order")
    thresholds = ordered["threshold_value"].astype(float).tolist()
    estimates = ordered["lr_estimate"].astype(float).tolist()
    fold_change = estimates[-1] / estimates[0]
    monotonic_class = classify_monotonicity(estimates)
    anchored_flag = bool(fold_change <= 1.10)
    return {
        "case_id": ordered["case_id"].iloc[0],
        "family": ordered["family"].iloc[0],
        "condition": ordered["condition"].iloc[0],
        "finding_family": ordered["finding_family"].iloc[0],
        "model": ordered["model"].iloc[0],
        "case_source": ordered["case_source"].iloc[0],
        "n_variants": len(ordered),
        "thresholds_tested": format_sequence(thresholds),
        "threshold_units": ordered["threshold_units"].iloc[0],
        "lr_estimates": format_sequence(estimates),
        "adjacent_differences": format_sequence(np.diff(estimates).tolist()),
        "monotonic_class": monotonic_class,
        "endpoint_fold_change": fold_change,
        "anchored_flag": anchored_flag,
        "min_lr": min(estimates),
        "max_lr": max(estimates),
    }

summary_df = pd.DataFrame(
    summarize_group(group)
    for _, group in raw_df.groupby(["case_id", "model"], sort=False)
)

summary_path = OUTPUT_DIR / "threshold_perturbation_summary.csv"
summary_df.to_csv(summary_path, index=False)

print(f"Saved summary -> {summary_path}")
display(summary_df)

## 6. Response-ready table and response text

These outputs are compact enough to paste into a supplement or response letter after reviewing the generated values.

In [ ]:
def interpret_summary_row(row: pd.Series) -> str:
    if row["monotonic_class"] == "flat":
        return "No threshold-related shift in the estimate."
    if row["anchored_flag"]:
        return "Only minimal endpoint change despite threshold perturbation."
    if row["monotonic_class"] == "strictly_increasing":
        return "Estimate increased as the threshold rose."
    if row["monotonic_class"] == "nondecreasing_with_ties":
        return "Estimate changed in the expected direction with at least one tie."
    return "Estimate did not change monotonically with the threshold."

response_table = summary_df[
    [
        "family",
        "condition",
        "model",
        "case_source",
        "thresholds_tested",
        "threshold_units",
        "lr_estimates",
        "monotonic_class",
        "endpoint_fold_change",
        "anchored_flag",
    ]
].copy()
response_table["endpoint_fold_change"] = response_table["endpoint_fold_change"].round(3)
response_table["interpretation"] = response_table.apply(interpret_summary_row, axis=1)

response_table_path = OUTPUT_DIR / "threshold_perturbation_response_table.csv"
response_table.to_csv(response_table_path, index=False)

n_families = summary_df["case_id"].nunique()
n_variants = len(raw_df)
n_monotonic = int(summary_df["monotonic_class"].isin(["strictly_increasing", "nondecreasing_with_ties", "flat"]).sum())
n_strict = int(summary_df["monotonic_class"].eq("strictly_increasing").sum())
n_nonmonotonic = int(summary_df["monotonic_class"].eq("nonmonotonic").sum())
n_anchored = int(summary_df["anchored_flag"].sum())
model_list = ", ".join(MODELS)

bullets = []
for row in response_table.itertuples(index=False):
    bullets.append(
        f"- {row.family}: thresholds {row.thresholds_tested} {row.threshold_units}; "
        f"GPT-5 LRs {row.lr_estimates}; monotonic class {row.monotonic_class}; "
        f"endpoint fold-change {row.endpoint_fold_change:.3f}; anchored flag {row.anchored_flag}."
    )

summary_text = f"""# Exploratory perturbation analysis of threshold-sensitive findings

We performed an exploratory perturbation analysis for threshold-dependent findings. We treat this as a sensitivity check rather than proof of reasoning, because monotonic changes in output cannot exclude memorization or distributional familiarity.

Methods: We selected {n_families} threshold-sensitive finding families and generated {n_variants} prompt variants by changing only the numeric threshold within otherwise similar finding text. The primary analysis queried {model_list} once per variant. The D-dimer family was included as a constructed constructed exemplar because the manuscript workbook includes D-dimer findings without numeric thresholds; the other families were grounded in threshold-dependent workbook findings.

Results: {n_monotonic} of {n_families} families were nondecreasing, flat, or strictly increasing across the tested thresholds, including {n_strict} strictly increasing families. {n_nonmonotonic} families were nonmonotonic. {n_anchored} families met the prespecified anchored flag, defined as an endpoint fold-change <= 1.10.

Family-level results:
"""
summary_text += "\n".join(bullets)
summary_text += "\n"

summary_md_path = OUTPUT_DIR / "threshold_perturbation_summary.md"
summary_md_path.write_text(summary_text, encoding="utf-8")

print(f"Saved response-ready table -> {response_table_path}")
print(f"Saved markdown summary -> {summary_md_path}")
display(response_table)
print(summary_text)

## 7. Run checks

Expected outputs after execution:

- `threshold_perturbation_cases.csv`: 15 rows.
- `threshold_perturbation_raw.csv`: 15 rows, all positive finite GPT-5 estimates.
- `threshold_perturbation_summary.csv`: 5 rows.
- `threshold_perturbation_response_table.csv`: 5 rows.
- `threshold_perturbation_summary.md`: includes the exploratory-analysis caveat.

In [ ]:
expected_files = {
    "cases": OUTPUT_DIR / "threshold_perturbation_cases.csv",
    "raw": OUTPUT_DIR / "threshold_perturbation_raw.csv",
    "summary": OUTPUT_DIR / "threshold_perturbation_summary.csv",
    "response_table": OUTPUT_DIR / "threshold_perturbation_response_table.csv",
    "markdown_summary": OUTPUT_DIR / "threshold_perturbation_summary.md",
}

for label, path in expected_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing expected {label} output: {path}")

checks = {
    "cases_rows": len(pd.read_csv(expected_files["cases"])),
    "raw_rows": len(pd.read_csv(expected_files["raw"])),
    "summary_rows": len(pd.read_csv(expected_files["summary"])),
    "response_table_rows": len(pd.read_csv(expected_files["response_table"])),
    "markdown_mentions_exploratory": "exploratory" in expected_files["markdown_summary"].read_text(encoding="utf-8").lower(),
}

if checks["cases_rows"] != 15:
    raise AssertionError(f"Expected 15 case rows, found {checks['cases_rows']}")
if checks["raw_rows"] != 15:
    raise AssertionError(f"Expected 15 raw rows, found {checks['raw_rows']}")
if checks["summary_rows"] != 5:
    raise AssertionError(f"Expected 5 summary rows, found {checks['summary_rows']}")
if checks["response_table_rows"] != 5:
    raise AssertionError(f"Expected 5 response table rows, found {checks['response_table_rows']}")
if not checks["markdown_mentions_exploratory"]:
    raise AssertionError("Markdown summary does not include the exploratory-analysis caveat.")

raw_check = pd.read_csv(expected_files["raw"])
if raw_check["error_status"].ne("ok").any():
    raise AssertionError("Raw output contains failed API calls.")
if (~np.isfinite(raw_check["lr_estimate"]) | (raw_check["lr_estimate"] <= 0)).any():
    raise AssertionError("Raw output contains non-positive or non-finite LR estimates.")

summary_check = pd.read_csv(expected_files["summary"])
if summary_check["monotonic_class"].isna().any() or summary_check["anchored_flag"].isna().any():
    raise AssertionError("Summary output has missing monotonicity or anchored flags.")

print(json.dumps(checks, indent=2))